# ***Setup and load data***

In [1]:
import json
import re
import statistics
from collections import Counter
from pathlib import Path

DATA_DIR = Path("../data")
SUB_DIR = Path("../submissions")

CANDIDATES = {
    "deep_agents": ["submission_deep_agents.json"],
    "deep_searcher": ["submission.json", "submission_deep.json"],
    "legal_graph": ["submission_legal_graph.json", "submission_graph.json"],
}

SUBMISSIONS = {}
for method, names in CANDIDATES.items():
    for name in names:
        if (SUB_DIR / name).exists():
            SUBMISSIONS[method] = SUB_DIR / name
            break

corpus_docs = json.load(open(DATA_DIR / "corpus_law_pub.json", encoding="utf-8"))

number_to_aid = {}
valid_keys = set()
corpus_law_ids = set()
for law in corpus_docs:
    law_id = str(law.get("law_id", "")).strip()
    corpus_law_ids.add(law_id)
    order = 0
    for article in law.get("content", []):
        order += 1
        number_to_aid[(law_id, order)] = article["aid"]
        valid_keys.add((law_id, article["aid"]))

test_cases = json.load(open(DATA_DIR / "ALQAC2026_public_test.json", encoding="utf-8"))
gold = {case["case_id"]: case for case in test_cases}

PREDICTIONS = {
    method: {record["case_id"]: record for record in json.load(open(path, encoding="utf-8"))}
    for method, path in SUBMISSIONS.items()
}

_CODE_RE = re.compile(r"\d+/\d{4}/[A-Za-zĐđ\-]+")
_ART_RE = re.compile(r"[Đđ]iều\s+(\d+)")
_OUTDATED = ("1987", "1993", "1995", "1998", "2000", "2003", "2004", "2005", "2006", "2009")


def resolve_law_id(name):
    for match in _CODE_RE.finditer(name):
        if match.group(0) in corpus_law_ids:
            return match.group(0)
    low = name.lower()
    outdated = any(year in low for year in _OUTDATED)
    if "tố tụng dân sự" in low:
        return "92/2015/QH13"
    if "tố tụng hành chính" in low:
        return "93/2015/QH13"
    if "dân sự" in low:
        return None if outdated else "91/2015/QH13"
    if "hình sự" in low:
        return "100/2015/QH13"
    if "đất đai" in low:
        return None if outdated else "45/2013/QH13"
    if "hôn nhân" in low:
        return None if outdated else "52/2014/QH13"
    if "án phí" in low or "lệ phí" in low:
        return None if "pháp lệnh" in low else "326/2016/UBTVQH14"
    if "thi hành án" in low:
        return "26/2008/QH12"
    return None


def gold_law_keys(record):
    keys = set()
    unresolved = 0
    for line in str(record.get("related_law_provisions", "")).splitlines():
        if "|" not in line:
            continue
        name, remainder = line.split("|", 1)
        law_id = resolve_law_id(name.strip())
        if law_id is None:
            unresolved += 1
            continue
        for raw_number in _ART_RE.findall(remainder):
            aid = number_to_aid.get((law_id, int(raw_number)))
            if aid is not None and (law_id, aid) in valid_keys:
                keys.add((law_id, aid))
            else:
                unresolved += 1
    return keys, unresolved


def predicted_law_keys(record):
    keys = set()
    for entry in (record or {}).get("law_evidence", []) or []:
        keys.add((str(entry.get("law_id", "")), int(entry.get("aid", -1))))
    return keys


print("methods:", {method: path.name for method, path in SUBMISSIONS.items()})
print("gold cases:", len(gold), "| corpus articles:", len(valid_keys))
print("note: public test has NO gold case-evidence segments; penalized case recall is scored server-side.")

methods: {'deep_agents': 'submission_deep_agents.json', 'deep_searcher': 'submission.json', 'legal_graph': 'submission_graph.json'}
gold cases: 50 | corpus articles: 3352
note: public test has NO gold case-evidence segments; penalized case recall is scored server-side.


# ***Outcome accuracy***

In [2]:
print(f"{'method':<15}{'accuracy':>9}{'n':>5}")
for method, preds in PREDICTIONS.items():
    per_label = Counter()
    per_correct = Counter()
    correct = 0
    total = 0
    for case_id, gold_case in gold.items():
        label = gold_case.get("verdict_label")
        if label is None:
            continue
        total += 1
        per_label[label] += 1
        prediction = (preds.get(case_id) or {}).get("prediction")
        if prediction == label:
            correct += 1
            per_correct[label] += 1
    accuracy = correct / total if total else 0.0
    print(f"{method:<15}{accuracy:>9.3f}{total:>5}")
    for label in sorted(per_label):
        print(f"    {label:<18}{per_correct[label]}/{per_label[label]}")

method          accuracy    n
deep_agents        0.600   50
    A_WIN             8/16
    B_WIN             9/10
    PARTIAL_A_WIN     10/19
    PARTIAL_B_WIN     3/5
deep_searcher      1.000   50
    A_WIN             16/16
    B_WIN             10/10
    PARTIAL_A_WIN     19/19
    PARTIAL_B_WIN     5/5
legal_graph        0.500   50
    A_WIN             11/16
    B_WIN             6/10
    PARTIAL_A_WIN     7/19
    PARTIAL_B_WIN     1/5


# ***Law evidence micro-F1***

In [3]:
print(f"{'method':<15}{'precision':>10}{'recall':>8}{'micro_f1':>10}{'unresolved':>12}")
for method, preds in PREDICTIONS.items():
    true_positive = 0
    false_positive = 0
    false_negative = 0
    unresolved_total = 0
    for case_id, gold_case in gold.items():
        gold_keys, unresolved = gold_law_keys(gold_case)
        unresolved_total += unresolved
        pred_keys = predicted_law_keys(preds.get(case_id))
        true_positive += len(gold_keys & pred_keys)
        false_positive += len(pred_keys - gold_keys)
        false_negative += len(gold_keys - pred_keys)
    precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
    micro_f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    print(f"{method:<15}{precision:>10.3f}{recall:>8.3f}{micro_f1:>10.3f}{unresolved_total:>12}")

method          precision  recall  micro_f1  unresolved
deep_agents         0.130   0.160     0.143         173
deep_searcher       0.972   1.000     0.986         173
legal_graph         0.139   0.070     0.093         173


# ***Case evidence sanity checks***

In [4]:
print(f"{'method':<15}{'empty':>6}{'mean':>7}{'median':>8}{'min':>5}{'max':>5}{'valid%':>8}{'dup_cases':>10}")
for method, preds in PREDICTIONS.items():
    counts = []
    empties = 0
    valid_segments = 0
    total_segments = 0
    duplicate_cases = 0
    for case_id in gold:
        evidence = (preds.get(case_id) or {}).get("case_evidence", []) or []
        counts.append(len(evidence))
        if not evidence:
            empties += 1
        if len(evidence) != len(set(evidence)):
            duplicate_cases += 1
        for segment in evidence:
            total_segments += 1
            if str(segment).startswith(case_id):
                valid_segments += 1
    valid_pct = 100 * valid_segments / total_segments if total_segments else 0.0
    print(
        f"{method:<15}{empties:>6}{statistics.mean(counts):>7.1f}{statistics.median(counts):>8.0f}"
        f"{min(counts):>5}{max(counts):>5}{valid_pct:>8.1f}{duplicate_cases:>10}"
    )

method          empty   mean  median  min  max  valid% dup_cases
deep_agents         0   18.6      20    2   33   100.0         0
deep_searcher       0    5.6       5    1   11   100.0         0
legal_graph         0    5.7       6    2   10   100.0         0


# ***API efficiency penalty analysis***

In [5]:
ASSUMED_GOLD_N = [3, 5, 8]


def efficiency_factor(calls, gold_n):
    n = max(gold_n, 1)
    if calls <= 2 * n:
        return 1.0
    if calls >= 5 * n:
        return 0.0
    return 1 - (calls - 2 * n) / (3 * n)


print("penalized_case_recall = raw_recall * efficiency_factor(calls, gold_n)")
print("E = 1 if calls <= 2n | ramp | 0 if calls >= 5n  (n = distinct gold relevant segments)")
print("calls estimated as distinct submitted D (optimistic) and 2*max(D,3) (penalty-safe budget upper bound)")
for method, preds in PREDICTIONS.items():
    distinct = [len((preds.get(case_id) or {}).get("case_evidence", []) or []) for case_id in gold]
    print(f"\n{method}: mean distinct D={statistics.mean(distinct):.1f} max D={max(distinct)}")
    header = f"  {'gold_n':>7}{'E(c=D)':>9}{'E(c=2D)':>10}{'%E=0(c=D)':>12}{'%E=0(c=2D)':>13}"
    print(header)
    for gold_n in ASSUMED_GOLD_N:
        e_optimistic = [efficiency_factor(d, gold_n) for d in distinct]
        e_budget = [efficiency_factor(2 * max(d, 3), gold_n) for d in distinct]
        zero_optimistic = 100 * sum(1 for value in e_optimistic if value == 0.0) / len(distinct)
        zero_budget = 100 * sum(1 for value in e_budget if value == 0.0) / len(distinct)
        print(
            f"  {gold_n:>7}{statistics.mean(e_optimistic):>9.3f}{statistics.mean(e_budget):>10.3f}"
            f"{zero_optimistic:>12.1f}{zero_budget:>13.1f}"
        )

penalized_case_recall = raw_recall * efficiency_factor(calls, gold_n)
E = 1 if calls <= 2n | ramp | 0 if calls >= 5n  (n = distinct gold relevant segments)
calls estimated as distinct submitted D (optimistic) and 2*max(D,3) (penalty-safe budget upper bound)

deep_agents: mean distinct D=18.6 max D=33
   gold_n   E(c=D)   E(c=2D)   %E=0(c=D)   %E=0(c=2D)
        3    0.167     0.142        80.0         84.0
        5    0.403     0.157        20.0         84.0
        8    0.800     0.240         0.0         54.0

deep_searcher: mean distinct D=5.6 max D=11
   gold_n   E(c=D)   E(c=2D)   %E=0(c=D)   %E=0(c=2D)
        3    0.913     0.476         0.0         20.0
        5    0.996     0.837         0.0          0.0
        8    1.000     0.975         0.0          0.0

legal_graph: mean distinct D=5.7 max D=10
   gold_n   E(c=D)   E(c=2D)   %E=0(c=D)   %E=0(c=2D)
        3    0.927     0.416         0.0         22.0
        5    1.000     0.829         0.0          0.0
        8    1.0

# ***Diagnosis summary***

In [6]:
print("Diagnosis")
print("=" * 60)
print("1. No gold case-evidence segments exist locally, so the exact penalized")
print("   case recall cannot be reproduced offline; it is graded server-side.")
print("2. Penalized case recall = raw_recall * E(calls, gold_n). E drops to 0")
print("   once the number of API calls reaches 5 * gold_n.")
print("3. Gold relevant segments per case are few (about 3-8). A pipeline that")
print("   submits many distinct segments (large mean D) also issues many API")
print("   calls, so E collapses toward 0 and penalized recall stays low even")
print("   when raw recall is high.")
print("4. Compare the mean D above: methods with small D keep E near 1, while a")
print("   method with large D is penalized regardless of retrieval quality.")
print("5. Fix: retrieve fewer, higher-precision segments and cap API calls to")
print("   about 2 * gold_n (small; roughly <= 6-8 distinct segments per case).")

Diagnosis
1. No gold case-evidence segments exist locally, so the exact penalized
   case recall cannot be reproduced offline; it is graded server-side.
2. Penalized case recall = raw_recall * E(calls, gold_n). E drops to 0
   once the number of API calls reaches 5 * gold_n.
3. Gold relevant segments per case are few (about 3-8). A pipeline that
   submits many distinct segments (large mean D) also issues many API
   calls, so E collapses toward 0 and penalized recall stays low even
   when raw recall is high.
4. Compare the mean D above: methods with small D keep E near 1, while a
   method with large D is penalized regardless of retrieval quality.
5. Fix: retrieve fewer, higher-precision segments and cap API calls to
   about 2 * gold_n (small; roughly <= 6-8 distinct segments per case).
